# Laboratorium 3: Od tokenów do reprezentacji wektorowych

W tym laboratorium porównamy rzadkie i gęste reprezentacje wektorowe oraz sprawdzimy, jak zachowują się w zadaniach opartych na podobieństwie tekstów.

Porównujemy:

- TF-IDF,
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`,
- `intfloat/multilingual-e5-small`.

Główne pytanie laboratorium:

**Czy bliskość dwóch tekstów w przestrzeni wektorowej odpowiada podobieństwu, którego potrzebujemy w konkretnym zadaniu NLP?**


## Po tych zajęciach student potrafi

- wyjaśnić różnicę między rzadką i gęstą reprezentacją wektorową;
- obliczyć i zinterpretować podobieństwo cosinusowe;
- wygenerować reprezentacje tekstu za pomocą modelu embeddingowego;
- zaimplementować wyszukiwanie najbliższych sąsiadów;
- ocenić jakość lokalnego sąsiedztwa;
- zbudować prosty benchmark wyszukiwania;
- zastosować PCA do analizy przestrzeni reprezentacji;
- rozpoznać przypadki, w których wysokie podobieństwo wektorowe nie odpowiada wymaganiom zadania;
- porównać jakość reprezentacji z jej kosztem obliczeniowym;
- uzasadnić wybór reprezentacji dla konkretnego zastosowania.


## Dane

Pracujemy na zbiorze PolEmo 2.0 wykorzystywanym w poprzednich laboratoriach.

Jeżeli istnieje plik:

```text
artifacts/polemo2_clean.parquet
```

należy użyć tej wersji danych. Jeśli pliku nie ma, proszę wrócić do Lab 1 i go przygotować.

W tym laboratorium nie używamy zbioru testowego do wyboru reprezentacji ani do strojenia eksperymentów.


## Ustawienia wstępne

Wymagane biblioteki:

```bash
pip install pandas numpy matplotlib scikit-learn pyarrow sentence-transformers
```

Kod startowy obejmuje wyłącznie importy, stałe i wczytanie danych. Implementacja eksperymentów należy do Państwa.


In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

from sentence_transformers import SentenceTransformer

RANDOM_STATE = 42
SAMPLE_SIZE = 1000

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

MODEL_A = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODEL_B = "intfloat/multilingual-e5-small"

pd.set_option("display.max_colwidth", 180)


In [ ]:
DATA_PATH = ARTIFACTS / "polemo2_clean.parquet"
assert DATA_PATH.exists(), "Brak artifacts/polemo2_clean.parquet. Proszę wykonać Lab 1."

data = pd.read_parquet(DATA_PATH).dropna(subset=["text", "target"]).reset_index(drop=True)
assert set(data["target"]) == {"plus", "minus", "zero", "amb"}

train_df = data[data["split"] == "train"].reset_index(drop=True)  # tylko train, bez validation i test
print(train_df.shape)
display(train_df.head())

# Zadanie 1: TF-IDF i podobieństwo cosinusowe

Proszę rozpocząć od tekstów:

```text
A = "hotel czysty wygodny"
B = "hotel bardzo czysty"
C = "samochód szybki sportowy"
```

Proszę zaimplementować:

1. reprezentację TF-IDF;
2. tabelę wartości TF-IDF;
3. macierz podobieństwa cosinusowego;
4. wskazanie najbardziej podobnej pary.

Następnie proszę powtórzyć eksperyment dla:

```text
A = "Hotel był bardzo dobry."
B = "Obiekt noclegowy zrobił na mnie świetne wrażenie."
C = "Hotel był bardzo zły."
```

Przed uruchomieniem kodu proszę zapisać, która para powinna być najbardziej podobna według TF-IDF, a która według człowieka.


In [ ]:
# Proszę zaimplementować rozwiązanie zadania 1.

# Zadanie 2: Gęste reprezentacje wektorowe

Proszę wczytać modele `MODEL_A` i `MODEL_B` oraz wygenerować reprezentacje trzech zdań z drugiej części zadania 1.

Dla obu modeli proszę:

1. sprawdzić wymiar wektora;
2. sprawdzić typ danych;
3. wygenerować reprezentacje z normalizacją;
4. obliczyć macierz podobieństwa cosinusowego;
5. porównać kolejność podobieństw z wynikiem TF-IDF.

Model E5 wymaga prefiksów. Reguła (z karty modelu): zadania symetryczne (podobieństwo dwóch tekstów, sąsiedzi, PCA) → `query: ...` po obu stronach; wyszukiwanie (Zadanie 6) → `query: ...` dla zapytania i `passage: ...` dla dokumentów.

W tym zadaniu porównujemy zdania między sobą, więc proszę użyć `query: ...` dla wszystkich trzech zdań.

Dla obu modeli proszę dodatkowo sprawdzić `model.max_seq_length` oraz, dla tekstów z próbki z Zadania 3, jaki odsetek jest dłuższy niż okno modelu (po tokenizacji). Jak to może wpływać na kolejne zadania?

Wynik proszę zebrać w krótkiej tabeli porównawczej.


In [ ]:
# Proszę wczytać oba modele i przeprowadzić porównanie.

# Podpowiedź: długości po tokenizacji
# lens = [len(model.tokenizer(t)["input_ids"]) for t in texts]
# (np.array(lens) > model.max_seq_length).mean()

# Zadanie 3: Próbka danych i koszt reprezentacji

Proszę przygotować deterministyczną próbkę `SAMPLE_SIZE` dokumentów ze zbioru treningowego. Próbka powinna w przybliżeniu zachowywać proporcje klas.

Następnie proszę utworzyć dla tej samej próbki:

- TF-IDF z `ngram_range=(1, 2)`, `min_df=2`, `max_features=50_000`;
- reprezentacje MiniLM;
- reprezentacje E5 z prefiksem `query: ` (wersja symetryczna, używana w Zadaniach 4, 5, 7 i 8; dla Zadania 6 dokumenty trzeba będzie zakodować ponownie z prefiksem `passage: `, ten czas nie wchodzi do tabeli).

Przy pomiarze czasu proszę wyłączyć ładowanie modelu, zrobić jedną rozgrzewkę (pojedyncze uruchomienie) na małej partii i zapisać urządzenie (CPU/GPU).

Dla każdej reprezentacji proszę zmierzyć:

- czas utworzenia;
- średni czas na dokument;
- liczbę wymiarów;
- typ reprezentacji: rzadka lub gęsta;
- rozmiar reprezentacji w pamięci.

Dla macierzy rzadkiej należy uwzględnić rzeczywisty koszt struktury sparse, a nie rozmiar macierzy po konwersji do postaci gęstej.

Wynik:

| reprezentacja | czas [s] | ms/dokument | liczba wymiarów | rozmiar [MB] | typ |
|---|---:|---:|---:|---:|---|
| TF-IDF | | | | | |
| MiniLM | | | | | |
| E5 | | | | | |


In [ ]:
# Proszę przygotować próbkę oraz zaimplementować benchmark reprezentacji.

# Zadanie 4: Najbliżsi sąsiedzi

Proszę zaimplementować funkcję:

```python
get_neighbours(embeddings, row_id, k=5)
```

która zwraca identyfikatory najbliższych dokumentów oraz wartość podobieństwa. Funkcja nie może zwracać samego dokumentu jako sąsiada. Proszę pobrać k+1 sąsiadów i usunąć z wyniku indeks równy row_id, nie zakładając, że jest on pierwszy.

Proszę wybrać trzy dokumenty z różnych klas i dla każdego porównać pięciu najbliższych sąsiadów (teksty proszę skrócić do pierwszych 200 znaków) dla:
- TF-IDF,
- MiniLM,
- E5.

Dla każdego dokumentu proszę krótko zaznaczyć:

- czy sąsiedzi są podobni tematycznie;
- czy mają tę samą etykietę sentymentu;
- czy podobieństwo wynika głównie ze wspólnego słownictwa.

In [ ]:
# Proszę zaimplementować wyszukiwanie najbliższych sąsiadów.

# Zadanie 5: Czystość sąsiedztwa

Dla dokumentu $x$:

$$
\text{purity@}k(x) = \frac{\text{liczba sąsiadów z tą samą etykietą}}{k}
$$

proszę policzyć średnią tej wartości po całej próbce (bez samego dokumentu jako sąsiada) dla k = 1, 5, 10 dla TF-IDF, MiniLM i E5. Dodatkowo proszę podać:

- punkt odniesienia: oczekiwaną purity dla losowych sąsiadów, czyli sumę kwadratów udziałów klas (`(train_df["target"].value_counts(normalize=True) ** 2).sum()`; przy rozkładzie z Lab 1 ok. 0,28);
- `purity@5` osobno dla każdej klasy.

Proszę policzyć:

- `purity@1`,
- `purity@5`,
- `purity@10`

dla TF-IDF, MiniLM i E5.

Wynik:

| reprezentacja | purity@1 | purity@5 | purity@10 |
|---|---:|---:|---:|
| TF-IDF | | | |
| MiniLM | | | |
| E5 | | | |


In [ ]:
# Proszę zaimplementować obliczanie purity@k.

# Zadanie 6: Miniaturowy benchmark wyszukiwania

Proszę przygotować osiem zapytań: po dwa dla każdej klasy `plus`, `minus`, `zero`, `amb`.

Dla każdego zapytania należy zdefiniować oczekiwaną klasę dokumentów. Dla klas `zero` i `amb` proszę wcześniej zapisać, co w tym zbiorze znaczy "dokument relewantny" (pomoże przejrzenie kilkunastu losowych dokumentów tych klas z próbki).

Dla E5 dokumenty z próbki proszę zakodować ponownie z prefiksem `passage: ` (zapytania z `query: `).

Wyniki proszę podać również per zapytanie. Osiem zapytań to bardzo mała próba, więc różnic mniejszych niż ok. 0,1 między reprezentacjami nie należy interpretować jako przewagi.

Następnie proszę zaimplementować wyszukiwanie dla:

- TF-IDF;
- MiniLM;
- E5.

Dla E5 zapytanie powinno być kodowane z prefiksem:

```text
query: ...
```

a dokumenty z prefiksem:

```text
passage: ...
```

Proszę zaimplementować i policzyć:

- `Precision@5`;
- `Precision@10`.

Wynik:

| reprezentacja | Precision@5 | Precision@10 |
|---|---:|---:|
| TF-IDF | | |
| MiniLM | | |
| E5 | | |

Ten benchmark jest uproszczony: etykieta sentymentu pełni rolę przybliżonej oceny relewancji.


In [ ]:
# Proszę przygotować zapytania, wyszukiwanie i funkcję precision_at_k(...).

# Zadanie 7: Wizualizacja reprezentacji

Dla reprezentacji E5 proszę zastosować PCA i zredukować wektory do dwóch wymiarów.

Proszę:

1. narysować wykres punktowy z kolorem odpowiadającym klasie;
2. podać udział wariancji wyjaśnionej przez dwie pierwsze składowe;
3. wskazać na wykresie lub opisać obszary, w których klasy wyraźnie się mieszają.

Nie należy traktować wykresu dwuwymiarowego jako pełnego obrazu oryginalnej przestrzeni.


In [ ]:
# Proszę zaimplementować PCA i przygotować wykres.

# Zadanie 8: Analiza przypadków

Proszę przygotować tabelę zawierającą dokładnie 12 przypadków, które pokazują ograniczenia reprezentacji wektorowych.

Pierwsze 6 przypadków należy znaleźć w wynikach najbliższych sąsiadów lub wyszukiwania. Powinny to być przykłady, w których podobieństwo jest wysokie, ale wynik jest problematyczny z punktu widzenia sentymentu lub intencji zapytania.

Kolejne 6 przypadków należy przygotować jako pary kontrastowe, np.:

```text
Hotel jest dobry.
Hotel nie jest dobry.
```

Dla każdej pary proszę policzyć podobieństwo w MiniLM i E5.

Tabela powinna zawierać:

| nr | typ | tekst A | tekst B / wynik | MiniLM | E5 | Δ MiniLM | Δ E5 | komentarz |
|---:|---|---|---|---:|---:|---:|---:|---|

Cosinusy E5 są skupione wysoko, więc nie wolno porównywać bezwzględnych wartości MiniLM i E5. Δ to różnica między cosinusem pary a cosinusem pary niezwiązanej tematycznie (np. "Hotel jest dobry." i "Samochód jest szybki.") liczonym tym samym modelem.

W komentarzu proszę używać krótkich kategorii, np.:

- negacja;
- wspólny temat;
- wspólne słownictwo;
- podobna forma;
- różny sentyment;
- bardzo krótki tekst;
- inna.


In [ ]:
# Proszę znaleźć 6 przypadków z danych, przygotować 6 par kontrastowych i zbudować tabelę 12 przypadków.

# Zadanie 9: Interpretacja wyników

To zadanie nie wymaga nowej implementacji.

Na podstawie wyników zadań 1-8 proszę odpowiedzieć:

1. W jakich sytuacjach TF-IDF zachowywał się lepiej od reprezentacji gęstych?
2. W jakich sytuacjach przewagę miały MiniLM lub E5?
3. Czy wysoka wartość `purity@k` oznacza, że reprezentacja jest ogólnie "lepsza semantycznie"?
4. Czy model o najlepszym `Precision@10` musi być najlepszym wyborem do klasyfikacji sentymentu?
5. Jakie dwa ograniczenia eksperymentu najbardziej wpływają na interpretację wyników?

Odpowiedzi powinny odwoływać się do uzyskanych liczb i przykładów.

# Zadanie 10: Decyzja projektowa

To zadanie nie wymaga nowej implementacji.

Proszę wybrać reprezentację dla każdego z trzech zastosowań:

1. szybkie wyszukiwanie po dokładnych nazwach i terminach;
2. wyszukiwanie semantyczne parafraz;
3. przyszły system RAG.

Dla każdego zastosowania proszę podać:

- wybraną reprezentację;
- jeden wynik ilościowy (jakość lub koszt) uzasadniający decyzję;
- jeden obserwowany przypadek błędu lub ograniczenie;
- koszt, który należy zaakceptować.

Na końcu proszę odpowiedzieć jednym akapitem na pytanie:

**Dlaczego embeddingu nie należy utożsamiać ze "znaczeniem tekstu"?**

# Karta modelu embeddingowego

Proszę wybrać jeden z dwóch modeli gęstej reprezentacji jako model rekomendowany do dalszych laboratoriów i utworzyć:

```text
artifacts/embedding_model_card.md
```

Karta powinna zawierać:

## Model
- nazwa checkpointu;
- wymiar reprezentacji;
- sposób normalizacji;
- wymagane prefiksy wejścia, jeśli dotyczy.

## Dane
- użyty zbiór;
- wielkość próbki;
- odsetek tekstów obciętych przez okno modelu;
- sposób doboru próbki.

## Koszt
- czas utworzenia reprezentacji;
- ms/dokument;
- rozmiar reprezentacji w pamięci.

## Jakość sąsiedztwa
- `purity@1`;
- `purity@5`;
- `purity@10`.

## Wyszukiwanie
- `Precision@5`;
- `Precision@10`.

## Analiza jakościowa
- najważniejsze obserwacje z PCA;
- dominujące typy błędów z analizy 20 przypadków.

## Zastosowanie
- do jakich zadań model jest rekomendowany;
- do jakich zadań nie należy go wybierać wyłącznie na podstawie tego laboratorium.

## Ograniczenia
- ograniczenia danych;
- ograniczenia benchmarku;
- ograniczenia samej reprezentacji.


# Wnioski

Na koniec pracy proszę zapisać krótkie wnioski. Każdy wniosek powinien wynikać z wykonanego eksperymentu i zawierać odwołanie do wyniku liczbowego, obserwacji albo konkretnego przypadku.

Nie należy przepisywać definicji TF-IDF, embeddingów ani podobieństwa cosinusowego.

# Warunki zaliczenia laboratorium

Laboratorium jest ukończone, jeśli zawiera:

- [ ] własną implementację eksperymentu TF-IDF i podobieństwa cosinusowego;
- [ ] reprezentacje MiniLM i E5;
- [ ] deterministyczną próbkę danych;
- [ ] benchmark czasu i pamięci trzech reprezentacji;
- [ ] implementację wyszukiwania najbliższych sąsiadów;
- [ ] `purity@1`, `purity@5`, `purity@10`;
- [ ] osiem zapytań w benchmarku wyszukiwania;
- [ ] własną implementację `Precision@k`;
- [ ] `Precision@5` i `Precision@10`;
- [ ] wizualizację PCA;
- [ ] tabelę 12 przypadków błędów i par kontrastowych;
- [ ] sprawdzenie okna modeli (`max_seq_length`) i odsetka obciętych tekstów;
- [ ] purity z punktem odniesienia (losowi sąsiedzi) i per klasa;
- [ ] odpowiedzi do zadania interpretacyjnego (Zadanie 9);
- [ ] plik `artifacts/embedding_model_card.md`;
- [ ] od pięciu do ośmiu wniosków końcowych.